In [ ]:
import os, json, socket, subprocess, datetime
res = {}
def tcp(host, port, t=5):
    try:
        s = socket.create_connection((host, port), timeout=t); s.close(); return 'OK'
    except Exception as e:
        return 'FAIL ' + repr(e)
res['tcp_8.8.8.8:53'] = tcp('8.8.8.8', 53)
res['tcp_8.8.8.8:443'] = tcp('8.8.8.8', 443)
res['tcp_1.1.1.1:443'] = tcp('1.1.1.1', 443)
res['tcp_140.82.112.3:443'] = tcp('140.82.112.3', 443)
try:
    r = subprocess.run(['cat','/etc/resolv.conf'], capture_output=True, text=True)
    res['etc_resolv_conf'] = r.stdout
except Exception as e:
    res['resolv_exc'] = repr(e)
try:
    with open('/etc/resolv.conf','w') as f:
        f.write('nameserver 1.1.1.1\nnameserver 8.8.8.8\n')
    r = subprocess.run(['cat','/etc/resolv.conf'], capture_output=True, text=True)
    res['resolv_conf_after'] = r.stdout
except Exception as e:
    res['resolv_write_exc'] = repr(e)
try:
    res['resolve_after_ollama.com'] = socket.gethostbyname('ollama.com')
except Exception as e:
    res['resolve_after_ollama.com'] = 'FAIL ' + repr(e)
try:
    r = subprocess.run(['curl','-s','-o','/dev/null','-w','%{http_code}','--max-time','20','https://ollama.com'], capture_output=True, text=True)
    res['curl_ollama_after'] = (r.stdout, r.stderr[:200])
except Exception as e:
    res['curl_after_exc'] = repr(e)
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    res['nvidia_smi'] = (r.returncode, (r.stdout or r.stderr)[:200])
except Exception as e:
    res['nvidia_exc'] = repr(e)
print(json.dumps(res, indent=2), flush=True)
with open('/kaggle/working/env.txt','w') as f:
    f.write(json.dumps(res, indent=2)+'\n')
print('done', flush=True)
